In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')

In [ ]:
df = pd.read_csv("./data/IMDB Dataset.csv")

In [ ]:
df

In [ ]:
df.sample(5)

In [ ]:
df_pos = df[df['sentiment']=='positive'][:5000]
df_neg = df[df['sentiment']=='negative'][:5000]

df_reviews = pd.concat([df_pos, df_neg ])

In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
train,test = train_test_split(df_reviews,test_size =0.33,random_state=42)

In [ ]:
train_x, train_y = train['review'], train['sentiment']
test_x, test_y = test['review'], test['sentiment']

In [ ]:
train_y.value_counts()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

train_x_vector = tfidf.fit_transform(train_x)
test_x_vector = tfidf.transform(test_x)

In [ ]:
train_x.shape

In [ ]:
train_x_vector.shape

In [ ]:
type(train_x_vector)

In [ ]:
primera_resenia = pd.DataFrame.sparse.from_spmatrix(train_x_vector,
                                  index=train_x.index,
                                  columns=tfidf.get_feature_names_out()).iloc[0]

In [ ]:
primera_resenia

In [ ]:
train_x.iloc[0]

In [ ]:
primera_resenia[primera_resenia != 0]

In [ ]:
train_x.iloc[0]

In [ ]:
from sklearn.svm import SVC
svc = SVC(kernel='linear')
svc.fit(train_x_vector, train_y)

In [ ]:
print(svc.predict(tfidf.transform(['A good movie'])))
print(svc.predict(tfidf.transform(['An excellent movie'])))
print(svc.predict(tfidf.transform(['I did not like this movie at all I gave this movie away'])))

In [ ]:
print(svc.score(test_x_vector, test_y))

In [ ]:
from sklearn.metrics import f1_score

f1_score(test_y,svc.predict(test_x_vector),
          labels = ['positive','negative'],average=None)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(test_y,
                            svc.predict(test_x_vector),
                            labels = ['positive','negative']))

In [ ]:
from sklearn.metrics import confusion_matrix

conf_mat = confusion_matrix(test_y,
                           svc.predict(test_x_vector),
                           labels = ['positive', 'negative'])
conf_mat

## Comparación con otros clasificadores

Para comprobar si la máquina de vectores de soporte fue la mejor elección, se entrenan tres modelos más con las mismas reseñas y los mismos vectores TF-IDF:

- `LogisticRegression`: clasificador lineal, el más cercano a una SVM de núcleo lineal en texto.
- `GaussianNB`: Naive Bayes gaussiano. Como no acepta matrices dispersas, cada lote de TF-IDF se convierte a una matriz densa.
- `DecisionTreeClassifier`: árbol de decisión, que separa las reseñas con reglas sobre las palabras.

La comparación usa el mismo conjunto de prueba y las mismas métricas: accuracy, F1 por clase, reporte de clasificación y matriz de confusión.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

def ajustar_gaussian_nb(X, y, batch_size=250):
    """GaussianNB no admite la matriz dispersa de TF-IDF."""
    modelo = GaussianNB()
    y_valores = y.to_numpy()
    clases = np.unique(y_valores)
    for inicio in range(0, X.shape[0], batch_size):
        fin = inicio + batch_size
        modelo.partial_fit(
            X[inicio:fin].toarray(),
            y_valores[inicio:fin],
            classes=clases,
        )
    return modelo

def predecir(modelo, X, denso=False, batch_size=250):
    if not denso:
        return modelo.predict(X)
    bloques = []
    for inicio in range(0, X.shape[0], batch_size):
        fin = inicio + batch_size
        bloques.append(modelo.predict(X[inicio:fin].toarray()))
    return np.concatenate(bloques)

logistic = LogisticRegression(max_iter=1000, random_state=42)
logistic.fit(train_x_vector, train_y)

gaussian_nb = ajustar_gaussian_nb(train_x_vector, train_y)

decision_tree = DecisionTreeClassifier(random_state=42)
decision_tree.fit(train_x_vector, train_y)

modelos = {
    "SVC (lineal)": (svc, False),
    "LogisticRegression": (logistic, False),
    "GaussianNB": (gaussian_nb, True),
    "DecisionTreeClassifier": (decision_tree, False),
}

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

frases = [
    "A good movie",
    "An excellent movie",
    "I did not like this movie at all I gave this movie away",
]
frases_vector = tfidf.transform(frases)

for nombre, (modelo, denso) in modelos.items():
    print(nombre, predecir(modelo, frases_vector, denso=denso))

filas = []
reportes = {}
matrices = {}

for nombre, (modelo, denso) in modelos.items():
    pred = predecir(modelo, test_x_vector, denso=denso)
    f1_pos, f1_neg = f1_score(
        test_y, pred, labels=["positive", "negative"], average=None
    )
    filas.append({
        "modelo": nombre,
        "accuracy": accuracy_score(test_y, pred),
        "f1_positive": f1_pos,
        "f1_negative": f1_neg,
        "f1_macro": (f1_pos + f1_neg) / 2,
    })
    reportes[nombre] = classification_report(
        test_y, pred, labels=["positive", "negative"]
    )
    matrices[nombre] = confusion_matrix(
        test_y, pred, labels=["positive", "negative"]
    )

comparacion = (
    pd.DataFrame(filas)
    .sort_values(["accuracy", "f1_macro"], ascending=False)
    .reset_index(drop=True)
)
comparacion

In [ ]:
for nombre in comparacion["modelo"]:
    print(f"===== {nombre} =====")
    print(reportes[nombre])
    print(matrices[nombre])
    print()

mejor = comparacion.iloc[0]
svm = comparacion.loc[comparacion["modelo"] == "SVC (lineal)"].iloc[0]
print(
    f"Mejor accuracy: {mejor['modelo']} "
    f"({mejor['accuracy']:.4f}), F1 macro {mejor['f1_macro']:.4f}."
)
print(
    f"SVC lineal: accuracy {svm['accuracy']:.4f}, "
    f"F1 macro {svm['f1_macro']:.4f}."
)

In [ ]:
comparacion_grafica = comparacion.melt(
    id_vars="modelo",
    value_vars=["accuracy", "f1_macro"],
    var_name="métrica",
    value_name="valor",
)

plt.figure(figsize=(9, 4.5))
sns.barplot(data=comparacion_grafica, x="modelo", y="valor", hue="métrica")
plt.ylim(0, 1)
plt.ylabel("Puntuación")
plt.xlabel("")
plt.title("Accuracy y F1 macro en el conjunto de prueba")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()